# IOAI — 2024 Final Stage Anomaly Detection (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/test.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2024-final-stage-anomaly-detection/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터 준비:', sorted(os.listdir('data'))[:8])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 이상 탐지 — 모범답안 (Detekcja anomalii, 사전학습 특징 + 근접이웃)

베이스라인(오토인코더 재구성오차, ≈0.82)보다 강한 접근이다. **동결된 사전학습 백본(ResNet18, ImageNet)**으로
이미지 임베딩을 뽑고, 각 테스트 이미지에서 **정상 학습 임베딩까지의 kNN 거리**를 이상 점수로 쓴다. 정상은
가깝고(거리 작음) 이상은 멀다(거리 큼). 임계값은 라벨 없이 **Otsu**(점수 분포의 골짜기)로 정한다.

제약 준수: 신경망(사전학습 백본) 사용·**정상 데이터만** 사용(이상 라벨 미사용, Otsu 는 점수 분포만 이용).
검증셋 accuracy ≈ **0.95**.

## 데이터

In [ ]:
import numpy as np, pandas as pd, torch, torch.nn as nn
from torchvision import models, transforms
from PIL import Image
device = "cuda" if torch.cuda.is_available() else "cpu"; print(device)

tf = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
def load(paths):
    return torch.stack([tf(Image.open(p).convert("RGB")) for p in paths])
train_files = ["data/train/"+f for f in pd.read_csv("data/train.csv")["filename"]]
test_df = pd.read_csv("data/test.csv"); test_files = ["data/test/"+f for f in test_df["filename"]]
print("train", len(train_files), "test", len(test_files))

## 사전학습 특징 추출

In [ ]:
# 동결 사전학습 백본으로 임베딩 추출(학습 안 함 — 정상만 참조로 사용)
bb = models.resnet18(weights=models.ResNet18_Weights.DEFAULT); bb.fc = nn.Identity()
bb.eval().to(device)
@torch.no_grad()
def feats(files):
    out=[]
    for i in range(0,len(files),128):
        out.append(bb(load(files[i:i+128]).to(device)).cpu().numpy())
    return np.concatenate(out)
Ftr = feats(train_files); Fte = feats(test_files)
Ftr /= np.linalg.norm(Ftr,axis=1,keepdims=True)+1e-8
Fte /= np.linalg.norm(Fte,axis=1,keepdims=True)+1e-8
print("features", Ftr.shape, Fte.shape)

## kNN 거리 + Otsu 임계값(numpy) → submission.csv

In [ ]:
from sklearn.neighbors import NearestNeighbors
# 이상 점수 = 정상 임베딩 중 최근접 k개까지의 평균 거리
knn = NearestNeighbors(n_neighbors=5).fit(Ftr)
dist,_ = knn.kneighbors(Fte); score = dist.mean(1)

def otsu_threshold(x, bins=256):
    """라벨 없이 점수 분포의 두 봉우리를 가르는 임계값(Otsu). numpy 만 사용."""
    hist, edges = np.histogram(x, bins=bins)
    centers = (edges[:-1] + edges[1:]) / 2
    w = hist.cumsum().astype(float); tot = w[-1]
    cum_mean = (hist * centers).cumsum(); mu_t = cum_mean[-1]
    wb = w / tot; wf = 1 - wb
    with np.errstate(divide='ignore', invalid='ignore'):
        mb = cum_mean / np.where(w == 0, np.nan, w)
        mf = (mu_t - cum_mean) / np.where(tot - w == 0, np.nan, tot - w)
        var_between = wb * wf * (mb - mf) ** 2
    return centers[np.nanargmax(var_between)]

thr = otsu_threshold(score)
pred = (score > thr).astype(int)
pd.DataFrame({"id": test_df["filename"], "label": pred}).to_csv("submission.csv", index=False)
print("saved submission.csv", len(pred), "| anomaly ratio", round(float(pred.mean()),3), "| thr", round(float(thr),4))

정상만으로 백본 특징을 자기지도(SimCLR 등)로 미세조정하면 도메인 특화 특징으로 더 오를 수 있다.

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)